# 01 — API Basics

**Objetivos:**
- Make first call to Claude API
- Understand messages structure (roles: user / assistant)
- Use system prompts
- Interpret response: `stop_reason`, `usage`, `content`

**Exam domain:** Context Management & Reliability (15%)

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

In [2]:
# Get the client and model from utils
from src.utils import get_client, get_model

client = get_client()
model = get_model()

## 1. First API Call

In [3]:
# Call the API
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is Claude? Answer in one sentence"
        }
    ]
)

In [4]:
# Complete response
message

Message(id='msg_01Qt8JShS5KgJ1hREyEfibpY', container=None, content=[TextBlock(citations=None, text='Claude is an AI assistant created by Anthropic to be helpful, harmless, and honest in conversations and tasks.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=27, server_tool_use=None, service_tier='standard'))

In [5]:
# Only the text content of the first message in the response
message.content[0].text

'Claude is an AI assistant created by Anthropic to be helpful, harmless, and honest in conversations and tasks.'

## 2. Multi-turn conversation

Claude doesn't have memory between calls. You must pass complete history between requests.

In [6]:
# Claude doesn't have memory
message1 = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is Claude? Answer in one sentence"
        }
    ]
)

message2 = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write another sentence"
        }
    ]
)

message1.content[0].text, message2.content[0].text

('Claude is an AI assistant created by Anthropic that can help with a wide variety of tasks like answering questions, writing, analysis, math, coding, and creative projects through text-based conversations.',
 'The morning sunlight filtered through the kitchen window, casting warm golden patterns across the wooden table where a steaming cup of coffee waited.')

In [3]:
# One way to give the model memory is to include the previous messages in the conversation
# See functions add_user_message, add_assistant_message, and chat in src/utils.py
from src.utils import chat, add_user_message, add_assistant_message

In [8]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "What is Claude? Answer in one sentence"
add_user_message(messages, "What is Claude? Answer in one sentence")

# Pass the list of messages into the chat to get a response
response = chat(messages, client, model)

# Take the answer and add it to the messages list as an assistant message
add_assistant_message(messages, response)

# Add in the user's follow up question
add_user_message(messages, "Write another sentence")

# Call chat again with the updated list of messages to get a response that takes into account the previous conversation
response = chat(messages, client, model)

response

'I can assist with a wide variety of tasks including writing, analysis, math, coding, creative projects, and answering questions on many topics.'

## 3. Exercise - Chatbot

Make a chat bot using the three helper functions we just put together.

1. Prompt the user to enter some input usint the built-in "input" function
2. Add it to a list of messages
3. Call the API
4. Add generated text to the list of messages
5. Print the generated text
6. Prepeat from #1

In [ ]:
# Make a starting list of messages
messages = []

MAX_MESSAGES = 10

while len(messages) < MAX_MESSAGES:
    # Get user input
    user_input = input("User: ")
    print(f"User: {user_input}")
    
    # Add user input to the list of messages
    add_user_message(messages, user_input)
    
    # Pass messages to the chat function to get a response
    response = chat(messages, client, model)
    
    # Add the response to the list of messages
    add_assistant_message(messages, response)
    print(f"Claude: {response}")

Claude: Claude is an AI assistant created by Anthropic to be helpful, harmless, and honest in conversations and tasks.
Claude: I'm designed to assist with a wide variety of tasks including answering questions, writing, analysis, math, coding, and creative projects.
Claude: I'm particularly useful for tasks that involve reasoning, analysis, and clear communication—such as explaining complex topics, helping with research and writing, problem-solving, coding assistance, and having thoughtful conversations about ideas.
Claude: 1+1 equals 2.
Claude: 1-1 equals 0.


## 4. Using system prompt

The `system prompt` provides Claude guidance on how to respond.

In [9]:
# Without system prompt
messages = []

add_user_message(messages, "How do I solve 5x+3=2 for x?")

response = chat(messages, client, model)

print(response)

To solve 5x + 3 = 2 for x, I need to isolate x by undoing the operations.

**Step 1:** Subtract 3 from both sides
5x + 3 - 3 = 2 - 3
5x = -1

**Step 2:** Divide both sides by 5
5x ÷ 5 = -1 ÷ 5
x = -1/5

**Answer:** x = -1/5 (or x = -0.2)

**Check:** Let me verify by substituting back into the original equation:
5(-1/5) + 3 = -1 + 3 = 2 ✓


In [ ]:
from src.utils import chat, add_user_message, add_assistant_message

In [10]:
# Math tutor system prompt
system_prompt = """
You are a patient math tutor.
Do not directly answer a student's question.
Guide them to a solution step by step.
"""

# With system prompt
messages = []

add_user_message(messages, "How do I solve 5x+3=2 for x?")

response = chat(messages, client, model, system_prompt)

print(response)

I'd be happy to help you solve this equation step by step! Let's work through it together.

First, let me ask you: what do you think our goal is when we're solving for x? What do we want x to look like when we're done?

Once you tell me that, we can think about what we need to do to the equation 5x + 3 = 2 to reach that goal.


## 5. Exercise - System prompts

Look at the following case:

In [12]:
messages = []

add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters."
)

response = chat(messages, client, model)

print(response)

Here are several Python functions to check for duplicate characters in a string:

## Method 1: Simple Boolean Check (Most Common)

```python
def has_duplicates(s):
    """
    Check if a string contains duplicate characters.
    Returns True if duplicates exist, False otherwise.
    """
    return len(s) != len(set(s))

# Example usage
print(has_duplicates("hello"))     # True (duplicate 'l')
print(has_duplicates("world"))     # True (duplicate 'l')
print(has_duplicates("python"))    # False (no duplicates)
```

## Method 2: Case-Insensitive Check

```python
def has_duplicates_ignore_case(s):
    """
    Check for duplicates ignoring case sensitivity.
    """
    s_lower = s.lower()
    return len(s_lower) != len(set(s_lower))

# Example usage
print(has_duplicates_ignore_case("Hello"))   # True ('l' appears twice)
print(has_duplicates_ignore_case("AaBb"))    # True (case-insensitive)
print(has_duplicates_ignore_case("Python"))  # False
```

## Method 3: Return Duplicate Characters

```

Claude generates a lot of text explaining the solution, and so on.

Generate a system prompt to make this as concise as possible.

In [14]:
system_prompt = """
You are a helpful programming assistant.
Write clear and efficient code to solve the user's problem.
Answer only with Python code, no explanations.
Answer with the best possible solution, not multiples steps or a first draft.
"""

messages = []

add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters."
)

response = chat(messages, client, model, system_prompt)

print(response)

```python
def has_duplicates(s):
    return len(s) != len(set(s))
```


## 6. Temperature

The temparature is a decimal value between 0 and 1, that influences the exact distributions of probabilities of the next word.

In [ ]:
from src.utils import chat, add_user_message, add_assistant_message

In [7]:
messages = []

add_user_message(
    messages,
    "Generate a one sentence movie idea"
)

response = chat(messages, client, model, temperature=1.0)

print(response)

A retired librarian discovers that every book she's ever checked out has secretly been rewriting itself to reflect the lives of its readers, and she must track down the most dangerous novel before its tragic ending becomes someone's reality.


## 7. Response streaming

The API responses may last a bit longer than what a user may be willing to wait. One solution is to put a spinner in the screen.

Other solution is to enable streaming of responses to perceive as if the "API were writing".

In [ ]:
# One option
messages = []

add_user_message(
    messages,
    "Write a 1 sentence description of a fake database"
)

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_013jDJfrtVv6L6Lrw8ZV63qx', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='The', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' "', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='Global', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDel

In [10]:
# Other option
messages = []

add_user_message(
    messages,
    "Write a 1 sentence description of a fake database"
)

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()

ParsedMessage(id='msg_0118ebfqZRir8CMG95zND6NV', container=None, content=[ParsedTextBlock(citations=None, text='FakeDB is a lightweight in-memory database simulator that generates realistic mock data with customizable schemas for testing and development purposes without requiring actual database connections.', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=34, server_tool_use=None, service_tier='standard'))

## 8. Structured data

It is common to generate pieces of structured data.

In [ ]:
# Claude uses to format the response to be printable in a markdown code block, so we have to remove the code block formatting to get the raw json
messages = []

add_user_message(
    messages,
    "Generate a very short event bridge rule as json"
)

chat(messages, client, model, temperature=0.0)

'```json\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n```'

In [3]:
from src.utils import chat, add_user_message, add_assistant_message

In [5]:
messages = []

add_user_message(
    messages,
    "Generate a very short event bridge rule as json"
)

# Already do the ```json part, so the model just has to fill in the rest of the json without adding any extra text
add_assistant_message(
    messages,
    "```json"
)

response = chat(messages, client, model, temperature=0.0, stop_sequences=["```"])
print(response)


{
  "Name": "OrderProcessingRule",
  "EventPattern": {
    "source": ["myapp.orders"],
    "detail-type": ["Order Placed"]
  },
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"
    }
  ]
}



In [ ]:
# Now the answer is completely legible by json parsers
import json

json.loads(response)

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

### Exercise

- Use message prefilling and stop sequences only to get three different commands in a single response
- Thre shouldn't be any comments or explanation
- Hint: message prefilling isn't limited to just characters like ```

In [10]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)

text = chat(messages, client, model, temperature=0.0)

text

'Here are three short AWS CLI commands:\n\n1. **List S3 buckets:**\n   ```bash\n   aws s3 ls\n   ```\n\n2. **Describe EC2 instances:**\n   ```bash\n   aws ec2 describe-instances\n   ```\n\n3. **Get caller identity:**\n   ```bash\n   aws sts get-caller-identity\n   ```'

In [8]:
from IPython.display import Markdown

Markdown(text)

Here are three short AWS CLI commands:

1. **List S3 buckets:**
   ```bash
   aws s3 ls
   ```

2. **Describe EC2 instances:**
   ```bash
   aws ec2 describe-instances
   ```

3. **Get caller identity:**
   ```bash
   aws sts get-caller-identity
   ```

In [15]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(messages, client, model, temperature=0.0, stop_sequences=["```"])

text.strip()

'aws s3 ls\naws ec2 describe-instances\naws iam list-users'